# CS502K — Magic Square Fill-in Puzzle

Given a partially filled 3x3 grid, fill in the blanks with unused numbers
from 1-9 so that every row, column, and diagonal sums to 15.


## Framework (copied from search4e.ipynb / AIMA Python)

**Where to find it:** open search4e.ipynb on GitHub → copy Problem, Node,
failure, cutoff, expand, path\_states, FIFOQueue, breadth\_first\_search.


In [5]:
import math
import heapq
from collections import deque, Counter
from itertools import permutations, combinations


# The base class — copied from search4e.ipynb
class Problem(object):
    def __init__(self, initial=None, goal=None, **kwds):
        self.__dict__.update(initial=initial, goal=goal, **kwds)

    def actions(self, state):        raise NotImplementedError
    def result(self, state, action): raise NotImplementedError
    def is_goal(self, state):        return state == self.goal
    def action_cost(self, s, a, s1): return 1
    def h(self, node):               return 0

    def __str__(self):
        return '{}({!r}, {!r})'.format(type(self).__name__, self.initial, self.goal)


# Node — copied from search4e.ipynb
class Node:
    def __init__(self, state, parent=None, action=None, path_cost=0):
        self.__dict__.update(state=state, parent=parent,
                             action=action, path_cost=path_cost)
    def __repr__(self): return '<{}>'.format(self.state)
    def __len__(self): return 0 if self.parent is None else (1 + len(self.parent))
    def __lt__(self, other): return self.path_cost < other.path_cost


failure = Node('failure', path_cost=math.inf)
cutoff  = Node('cutoff',  path_cost=math.inf)


# expand, path_states — copied from search4e.ipynb
def expand(problem, node):
    s = node.state
    for action in problem.actions(s):
        s1 = problem.result(s, action)
        cost = node.path_cost + problem.action_cost(s, action, s1)
        yield Node(s1, node, action, cost)


def path_states(node):
    if node in (cutoff, failure, None):
        return []
    return path_states(node.parent) + [node.state]


# BFS — copied from search4e.ipynb
FIFOQueue = deque

def breadth_first_search(problem):
    node = Node(problem.initial)
    if problem.is_goal(problem.initial):
        return node
    frontier = FIFOQueue([node])
    reached = {problem.initial}
    while frontier:
        node = frontier.pop()
        for child in expand(problem, node):
            s = child.state
            if problem.is_goal(s):
                return child
            if s not in reached:
                reached.add(s)
                frontier.appendleft(child)
    return failure


# Helper: display a 3x3 grid (blanks shown as _)
def board9(state):
    def cell(v): return '_' if v == 0 else str(v)
    return '{}  {}  {}\n{}  {}  {}\n{}  {}  {}\n'.format(*[cell(v) for v in state])


---
## Task 1 — Solve the Fill-in Puzzle (find Figure 1)

**Key differences from the SWAP version (Assignment 3):**

| | Swap version | Fill-in version |
|---|---|---|
| Initial state | all 9 numbers placed wrong | some numbers + blanks (0) |
| Actions | swap any two positions (36) | place an unused number in next blank |
| Result | positions of two numbers exchange | a blank gets filled |
| Depth of solution | few swaps | number of blanks to fill |

The puzzle is like the 8-puzzle but:
- Blanks (0) represent empty cells to fill
- Instead of sliding tiles, we place unused numbers into blanks
- Goal is Figure 1: (4, 9, 2, 3, 5, 7, 8, 1, 6)

We do NOT override is\_goal here — the default from Problem checks
state == self.goal, which is Figure 1.


In [6]:
class MagicSquareFill(Problem):

    def __init__(self, initial, goal=(4, 9, 2, 3, 5, 7, 8, 1, 6)):
        # goal = Figure 1, used by is_goal, h1 and h2
        super().__init__(initial=initial, goal=goal)

    def actions(self, state):
        # ◀◀ DIFFERENT from swap version
        # Swap version: return [(i, j) for i in range(9) for j in range(i+1, 9)]
        # Fill version: return which unused numbers can go in the next blank
        if 0 not in state:
            return []                  # no blanks left, no actions
        unused = [n for n in range(1, 10) if n not in state]
        return unused                  # each action = a number to place

    def result(self, state, action):
        # ◀◀ DIFFERENT fromn swap version
        # Swap version: swap state[i] and state[j]
        # Fill version: put the number in the first blank
        s = list(state)
        blank = s.index(0)             # find first blank (first 0)
        s[blank] = action              # place the number there
        return tuple(s)

    # is_goal is NOT overridden — uses default: state == self.goal (Figure 1)

    def h1(self, node):
        # Count positions that don't match Figure 1 (skip blanks)
        # ◀◀ DIFFERENT: we skip blanks (0) because they haven't been filled yet
        return sum(s != g for s, g in zip(node.state, self.goal) if s != 0)

    def h2(self, node):
        # Count how many blanks are left to fill
        # ◀◀ DIFFERENT: Manhattan distance makes no sense here because
        # tiles don't move — they get placed. So we count remaining blanks.
        return node.state.count(0)

    def h(self, node):
        return self.h2(node)


In [7]:
# Starting state: 4, 5, 2, 1 are placed, rest are blanks (0)
#   4  _  2
#   _  5  _
#   _  1  _

initial = (4, 0, 2, 0, 5, 0, 0, 1, 0)

p1 = MagicSquareFill(initial=initial)

print("Starting grid:")
print(board9(p1.initial))

solution = breadth_first_search(p1)

print("Solution found (Figure 1):")
print(board9(solution.state))
print("Steps needed:", len(solution))

print("Path from start to solution:")
for i, s in enumerate(path_states(solution)):
    print(f"Step {i}: placed {s}")
    print(board9(s))


Starting grid:
4  _  2
_  5  _
_  1  _

Solution found (Figure 1):
4  9  2
3  5  7
8  1  6

Steps needed: 5
Path from start to solution:
Step 0: placed (4, 0, 2, 0, 5, 0, 0, 1, 0)
4  _  2
_  5  _
_  1  _

Step 1: placed (4, 9, 2, 0, 5, 0, 0, 1, 0)
4  9  2
_  5  _
_  1  _

Step 2: placed (4, 9, 2, 3, 5, 0, 0, 1, 0)
4  9  2
3  5  _
_  1  _

Step 3: placed (4, 9, 2, 3, 5, 7, 0, 1, 0)
4  9  2
3  5  7
_  1  _

Step 4: placed (4, 9, 2, 3, 5, 7, 8, 1, 0)
4  9  2
3  5  7
8  1  _

Step 5: placed (4, 9, 2, 3, 5, 7, 8, 1, 6)
4  9  2
3  5  7
8  1  6



---
## Task 2 — Find Additional Solutions

We modify the puzzle class by overriding is\_goal to accept ANY arrangement
where all rows, columns, and diagonals sum to 15 AND there are no blanks left.
This lets us find all valid completions of the starting grid.


In [ ]:
# Modified class — overrides is_goal to accept any valid magic square
class MagicSquareFillV2(MagicSquareFill):

    def is_goal(self, state):
        # must have no blanks AND all lines sum to 15
        if 0 in state:
            return False
        return (
            state[0]+state[1]+state[2] == 15 and
            state[3]+state[4]+state[5] == 15 and
            state[6]+state[7]+state[8] == 15 and
            state[0]+state[3]+state[6] == 15 and
            state[1]+state[4]+state[7] == 15 and
            state[2]+state[5]+state[8] == 15 and
            state[0]+state[4]+state[8] == 15 and
            state[2]+state[4]+state[6] == 15
        )


# Find ALL valid completions using BFS (it will find the first one)
p2 = MagicSquareFillV2(initial=(4, 0, 2, 0, 5, 0, 0, 1, 0))
sol2 = breadth_first_search(p2)

print("Solution found by BFS:")
print(board9(sol2.state))
print("Is it Figure 1?", sol2.state == (4, 9, 2, 3, 5, 7, 8, 1, 6))


In [ ]:
# Find ALL possible completions by exploring all branches
# (since the search tree is small, we can check all permutations of unused numbers)
from itertools import permutations

initial = (4, 0, 2, 0, 5, 0, 0, 1, 0)
placed = {v: i for i, v in enumerate(initial) if v != 0}  # numbers already placed
blanks = [i for i, v in enumerate(initial) if v == 0]     # positions of blanks
unused = [n for n in range(1, 10) if n not in initial]    # numbers not yet placed

checker = MagicSquareFillV2(initial=initial)

all_solutions = []
for perm in permutations(unused):
    # fill blanks with this permutation
    grid = list(initial)
    for pos, val in zip(blanks, perm):
        grid[pos] = val
    state = tuple(grid)
    if checker.is_goal(state):
        all_solutions.append(state)

print(f"Total valid completions: {len(all_solutions)}\n")
for i, sol in enumerate(all_solutions, 1):
    print(f"Solution {i}: {sol}")
    print(board9(sol))


---
## Task 3 — Compare Search Algorithms

Using MagicSquareFillV2 (accepts any valid magic square), we run DFS, BFS,
and best-first (with h1 and h2) and report summary statistics.

**Where to find it:** search4e.ipynb → search for "CountCalls" and "report".
Also copy depth\_limited\_search and best\_first\_search.


In [ ]:
# DFS — copied from search4e.ipynb
LIFOQueue = list

def is_cycle(node, k=30):
    def find_cycle(ancestor, k):
        return (ancestor is not None and k > 0 and
                (ancestor.state == node.state or find_cycle(ancestor.parent, k - 1)))
    return find_cycle(node.parent, k)

def depth_limited_search(problem, limit=50):
    frontier = LIFOQueue([Node(problem.initial)])
    result = failure
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        elif len(node) >= limit:
            result = cutoff
        elif not is_cycle(node):
            for child in expand(problem, node):
                frontier.append(child)
    return result


# Best-first search — copied from search4e.ipynb
class PriorityQueue:
    def __init__(self, items=(), key=lambda x: x):
        self.key = key
        self.items = []
        for item in items:
            self.add(item)
    def add(self, item):
        heapq.heappush(self.items, (self.key(item), item))
    def pop(self):
        return heapq.heappop(self.items)[1]
    def __len__(self): return len(self.items)

def best_first_search(problem, f):
    node = Node(problem.initial)
    frontier = PriorityQueue([node], key=f)
    reached = {problem.initial: node}
    while frontier:
        node = frontier.pop()
        if problem.is_goal(node.state):
            return node
        for child in expand(problem, node):
            s = child.state
            if s not in reached or child.path_cost < reached[s].path_cost:
                reached[s] = child
                frontier.add(child)
    return failure


# CountCalls and report — copied from search4e.ipynb
class CountCalls:
    def __init__(self, obj):
        self._object = obj
        self._counts = Counter()
    def __getattr__(self, attr):
        self._counts[attr] += 1
        return getattr(self._object, attr)


def report(searchers, problems):
    for searcher in searchers:
        print(searcher.__name__ + ':')
        total_counts = Counter()
        for p in problems:
            prob   = CountCalls(p)
            soln   = searcher(prob)
            counts = prob._counts
            counts.update(actions=len(soln), cost=soln.path_cost)
            total_counts += counts
            report_counts(counts, str(p)[:40])
        report_counts(total_counts, 'TOTAL\n')


def report_counts(counts, name):
    print('{:9,d} nodes |{:9,d} goal |{:5.0f} cost |{:8,d} actions | {}'.format(
        counts['result'], counts['is_goal'],
        counts['cost'],   counts['actions'], name))


In [ ]:
# Wrappers so each algorithm has a clear name in the report
def dfs(problem): return depth_limited_search(problem, limit=50)
dfs.__name__ = 'depth_first_search'

def best_h1(problem): return best_first_search(problem, f=lambda n: problem.h1(n))
best_h1.__name__ = 'best_first_h1'

def best_h2(problem): return best_first_search(problem, f=lambda n: problem.h2(n))
best_h2.__name__ = 'best_first_h2'

# Run report
report(
    [breadth_first_search, dfs, best_h1, best_h2],
    [MagicSquareFillV2(initial=(4, 0, 2, 0, 5, 0, 0, 1, 0))]
)


---
## Task 4 — Top Three Learnings

### Learning 1 — The search tree is much smaller than the swap version

In the swap version, every state has 36 possible actions (swap any two tiles).
In the fill-in version, actions decrease as blanks get filled: the first blank
has at most 5 choices (unused numbers), the next has 4, then 3, etc. This means
the total search tree is at most 5! = 120 states — tiny compared to the swap
version which can explore millions of states.

### Learning 2 — h1 and h2 from EightPuzzle don't apply well here

The 8-puzzle h1 (misplaced tiles) and h2 (Manhattan distance) assume tiles
MOVE between positions. In the fill-in puzzle, tiles don't move — blanks get
filled with new numbers. Manhattan distance is meaningless because nothing
slides.

Our adapted heuristics are:
- h1: count filled positions that don't match Figure 1 (but this ignores blanks)
- h2: count remaining blanks (simpler but gives same value to all states at same depth)

A better heuristic would check partial sums: if a row already has 2 numbers
that sum to more than 15, no valid number can complete it, so we could prune
those branches early.

### Learning 3 — How you define actions changes the nature of the search

In the swap version, the initial state already has all 9 numbers — the search
rearranges them. In the fill-in version, the initial state is incomplete — the
search builds up the solution by adding one number at a time. This makes it
more like a constraint satisfaction problem than a traditional state-space search.
BFS and DFS both work efficiently here because the solution is always exactly
(number of blanks) steps deep.
